# Know Your Data

**Research question:** Are property prices in Mexico more influenced by **location** or **property size**?

Before analysis, we need to make the three files consistent and understand what is inside them.

### Tidy data
- Each variable → one column
- Each observation → one row
- Each value → one cell

In [94]:
import pandas as pd

## 1. Load the data

We have three CSV files. First, define their paths so we can reuse them.

In [95]:
path_1 = "../datasets/mexico1.csv"
path_2 = "../datasets/mexico2.csv"
path_3 = "../datasets/mexico3.csv"

## 2. Inspect the first file

Before cleaning anything, take a quick look at the data.

We want to answer:
- What columns are present?
- Are values stored in the right form?
- Are there missing values?
- How large is the file?

In [96]:
df_raw = pd.read_csv(path_1)

df_raw.head()

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.56,-99.23,150.00,"$67,965.56"
1,house,Nuevo León,25.69,-100.20,186.00,"$63,223.78"
2,apartment,Guerrero,16.77,-99.76,82.00,"$84,298.37"
3,apartment,Guerrero,16.83,-99.91,150.00,"$94,308.80"
4,house,Veracruz de Ignacio de la Llave,NaN,NaN,175.00,"$94,835.67"


In [97]:
df_raw.shape

(700, 6)

`.shape` returns **(rows, columns)**.

Here, `(700, 6)` means 700 listings and 6 columns.

In [98]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  700 non-null    str    
 1   state          700 non-null    str    
 2   lat            583 non-null    float64
 3   lon            583 non-null    float64
 4   area_m2        700 non-null    float64
 5   price_usd      700 non-null    str    
dtypes: float64(3), str(3)
memory usage: 32.9 KB


`.info()` gives a quick summary of the DataFrame.

Pay attention to:
- **non-null counts** → where values are missing
- **dtypes** → whether values are stored as numbers or text

For example, `price_usd` is text because it contains `$` and `,`.

In [99]:
df_raw.dtypes

property_type        str
state                str
lat              float64
lon              float64
area_m2          float64
price_usd            str
dtype: object

In [100]:
df_raw.isna().sum()

property_type      0
state              0
lat              117
lon              117
area_m2            0
price_usd          0
dtype: int64

### What we found

- `lat` and `lon` have missing values.
- `price_usd` is stored as text.
- The other columns look usable.

Now we can decide how to clean the file.

## 3. Clean file 1

### Goal

1. Remove rows with missing values.
2. Convert `price_usd` from text to a number.

We use **method chaining** so the cleaning steps read from top to bottom:

`read → remove missing rows → clean price`

In [101]:
df1 = (
    pd.read_csv(path_1)
    .dropna()
    .assign(
        price_usd=lambda x: (
            x["price_usd"]
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .astype(float)
        )
    )
)

`lambda x` means: **use the DataFrame as it exists at this point in the chain**.

For example:

`"$55,000.00"` → `"55,000.00"` → `"55000.00"` → `55000.0`

In [102]:
df1.dtypes

property_type        str
state                str
lat              float64
lon              float64
area_m2          float64
price_usd        float64
dtype: object

In [103]:
df1.isna().sum()

property_type    0
state            0
lat              0
lon              0
area_m2          0
price_usd        0
dtype: int64

In [104]:
df1.shape

(583, 6)

File 1 is now clean: no missing values remain and `price_usd` is numeric.

## 4. Clean file 2

File 2 has the same structure, but its price is stored in **Mexican pesos (`price_mxn`)**.

We convert it to USD using the project conversion rate of **19 MXN per USD**, then keep the same `price_usd` column used in file 1.

In [105]:
MXN_PER_USD = 19

df2 = (
    pd.read_csv(path_2)
    .dropna()
    .assign(
        price_usd=lambda x: x["price_mxn"].div(MXN_PER_USD)
    )
    .drop(columns="price_mxn")
)

In [106]:
df2.head()

,property_type,state,lat,lon,area_m2,price_usd
0,apartment,Nuevo León,25.72,-100.35,72.00,"68,421.05"
2,house,Morelos,23.63,-102.55,360.00,"278,947.37"
6,apartment,Estado de México,19.27,-99.57,85.00,"65,789.47"
7,house,San Luis Potosí,22.14,-101.00,158.00,"111,578.95"
8,apartment,Distrito Federal,19.39,-99.13,65.00,"39,904.74"


In [107]:
df2.dtypes

property_type        str
state                str
lat              float64
lon              float64
area_m2          float64
price_usd        float64
dtype: object

In [108]:
df2.isna().sum()

property_type    0
state            0
lat              0
lon              0
area_m2          0
price_usd        0
dtype: int64

In [109]:
df2.shape

(571, 6)

## 5. Clean file 3

File 3 stores some information differently:

- `lat-lon` contains latitude and longitude in one cell.
- `place_with_parent_names` contains the location hierarchy.

We split those values into separate columns so the final DataFrame has the same structure as files 1 and 2.

In [110]:
df3 = (
    pd.read_csv(path_3)
    .dropna()
    .assign(
        state=lambda x: x["place_with_parent_names"].str.split("|", expand=True)[2],
        lat=lambda x: x["lat-lon"].str.split(",", expand=True)[0].astype(float),
        lon=lambda x: x["lat-lon"].str.split(",", expand=True)[1].astype(float),
    )
    .drop(columns=["place_with_parent_names", "lat-lon"])
)

In [111]:
df3.head()

,property_type,area_m2,price_usd,state,lat,lon
0,apartment,71.00,"48,550.59",Distrito Federal,19.53,-99.15
1,house,233.00,"168,636.73",Estado de México,19.26,-99.57
2,house,300.00,"86,932.69",Estado de México,19.27,-99.67
4,apartment,84.00,"68,508.67",Veracruz de Ignacio de la Llave,19.51,-96.87
5,house,175.00,"102,763.00",Jalisco,20.69,-103.37


In [112]:
df3.dtypes

property_type        str
area_m2          float64
price_usd        float64
state                str
lat              float64
lon              float64
dtype: object

In [113]:
df3.isna().sum()

property_type    0
area_m2          0
price_usd        0
state            0
lat              0
lon              0
dtype: int64

In [114]:
df3.shape

(582, 6)

## 6. Make sure the three files match

Before combining them, check that they have the same columns.

This matters because `concat()` works best when the DataFrames have the same structure.

In [115]:
expected_columns = [
    "property_type",
    "state",
    "lat",
    "lon",
    "area_m2",
    "price_usd",
]

df1 = df1[expected_columns]
df2 = df2[expected_columns]
df3 = df3[expected_columns]

assert list(df1.columns) == expected_columns
assert list(df2.columns) == expected_columns
assert list(df3.columns) == expected_columns

print("All three DataFrames have the same structure.")

All three DataFrames have the same structure.


## 7. Combine the cleaned data

Now that all three files have the same structure, stack them row by row.

`ignore_index=True` gives the combined DataFrame a fresh index from 0.

In [116]:
df = pd.concat(
    [df1, df2, df3],
    ignore_index=True
)

In [117]:
df.head()

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.56,-99.23,150.00,"67,965.56"
1,house,Nuevo León,25.69,-100.20,186.00,"63,223.78"
2,apartment,Guerrero,16.77,-99.76,82.00,"84,298.37"
3,apartment,Guerrero,16.83,-99.91,150.00,"94,308.80"
4,house,Yucatán,21.05,-89.54,205.00,"105,191.37"


In [118]:
df.shape

(1736, 6)

In [119]:
df.isna().sum()

property_type    0
state            0
lat              0
lon              0
area_m2          0
price_usd        0
dtype: int64

## 8. Save the cleaned dataset

We now have one clean DataFrame ready for analysis.

`index=False` prevents pandas from saving the DataFrame index as an extra column.

In [120]:
output_path = "../datasets/mexico_clean.csv"

df.to_csv(output_path, index=False)

## 9. First statistical look

Before deeper analysis, look at the main numeric variables: `price_usd` and `area_m2`.

`.describe()` gives:
- `mean` → average
- `std` → spread
- `min` / `max` → smallest and largest
- `25%`, `50%`, `75%` → quartiles

In [121]:
summary = df[["price_usd", "area_m2"]].describe()

summary

,price_usd,area_m2
count,"1,736.00","1,736.00"
mean,"115,331.98",170.26
std,"65,426.17",80.59
min,"33,157.89",60.00
25%,"65,789.47",101.75
50%,"99,262.13",156.00
75%,"150,846.66",220.00
max,"326,733.66",385.00


### What should we notice?

For this dataset:

- **Price mean ≈ $115k**
- **Price median ≈ $99k**
- Mean is higher than median → prices are **right-skewed**.
- Price has a large standard deviation → prices vary considerably.
- **Area mean ≈ 170 m²**
- **Area median = 156 m²**

The summary gives us a first idea of the data, but it does not show the full distribution or relationships between variables.